# 10.10 SHAP and LIME

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.10_shap-and-lime.ipynb)

Companion code for Section 10.10 of the book. Run the cells in order from top to bottom.

**Example 10.10.1**

In [ ]:
# pip install shap lime
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = list(data.feature_names)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train model
model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                     max_depth=4, random_state=42)
model.fit(X_tr, y_tr)

# ── SHAP TreeExplainer (fast for tree-based models) ───────────────
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_te)   # Shape: (n_test, n_features)

print('=== SHAP Analysis ===')
print(f'SHAP values shape: {shap_values.shape}')
print(f'Expected value (baseline): {explainer.expected_value:.4f}')
print(f'For sample 0:')
print(f'  Actual prediction:  {model.predict_proba([X_te[0]])[0][1]:.4f}')
print(f'  SHAP explanation:   {explainer.expected_value + shap_values[0].sum():.4f}')
print(f'  (sum check passes: {np.allclose(explainer.expected_value + shap_values[0].sum(),
                                           model.predict_proba([X_te[0]])[0][1], atol=1e-4)})')

# SHAP feature importance (mean absolute SHAP value)
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print('\nSHAP Feature Importance (top 10):')
print(shap_importance.head(10).round(4).to_string(index=False))

# ── SHAP Visualisations ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('SHAP Explainability Analysis', fontsize=14, fontweight='bold')

# 1. Summary plot (beeswarm)
ax = axes[0, 0]
shap.summary_plot(shap_values, X_te, feature_names=feature_names,
                   show=False, max_display=15, plot_type='violin')
ax = plt.gca()
ax.set_title('SHAP Summary Plot\n(each point = one sample)')

# 2. Feature importance bar
ax = axes[0, 1]
shap_importance.head(12).sort_values('mean_abs_shap').plot(
    kind='barh', x='feature', y='mean_abs_shap',
    ax=ax, color='#FF6F00', edgecolor='white', legend=False)
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('SHAP Feature Importance\n(global explanation)')
ax.grid(alpha=0.3, axis='x')

# 3. Force plot for single prediction (waterfall style)
ax = axes[1, 0]
sample_idx = 0
sv = shap_values[sample_idx]
feature_vals = X_te[sample_idx]
baseline = explainer.expected_value

# Waterfall chart
contributions = list(zip(sv, feature_names, feature_vals))
contributions_sorted = sorted(contributions, key=lambda x: abs(x[0]), reverse=True)[:10]
svs, fns, fvs = zip(*contributions_sorted)

colors = ['#1976D2' if v > 0 else '#E53935' for v in svs]
y_pos  = range(len(svs))
ax.barh(y_pos, svs, color=colors, edgecolor='white', alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'{fn}\n(={fv:.3f})' for fn, fv in zip(fns, fvs)], fontsize=8)
ax.axvline(0, color='black', lw=1)
pred_val = model.predict_proba([X_te[sample_idx]])[0][1]
ax.set_title(f'SHAP Waterfall (Sample 0)\nPred={pred_val:.3f}  True={data.target_names[y_te[sample_idx]]}')
ax.set_xlabel('SHAP Value (contribution to prediction)')
ax.grid(alpha=0.3, axis='x')

# 4. Dependence plot
ax = axes[1, 1]
top_feat_idx = shap_importance.iloc[0]['feature']
top_feat_pos = feature_names.index(top_feat_idx)
ax.scatter(X_te[:, top_feat_pos], shap_values[:, top_feat_pos],
          c=X_te[:, top_feat_pos], cmap='RdYlBu_r', s=20, alpha=0.7)
ax.set_xlabel(top_feat_idx)
ax.set_ylabel(f'SHAP value for {top_feat_idx}')
ax.set_title(f'SHAP Dependence Plot:\n{top_feat_idx}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── LIME ─────────────────────────────────────────────────────────
try:
    import lime
    import lime.lime_tabular

    print('\n=== LIME Analysis ===')
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        X_tr,
        feature_names=feature_names,
        class_names=list(data.target_names),
        mode='classification'
    )

    # Explain a single prediction
    for sample_idx in [0, 3, 10]:
        lime_exp = lime_explainer.explain_instance(
            X_te[sample_idx],
            model.predict_proba,
            num_features=8,
            num_samples=1000
        )

        pred = model.predict([X_te[sample_idx]])[0]
        true = y_te[sample_idx]
        print(f'\nSample {sample_idx}: True={data.target_names[true]}  Pred={data.target_names[pred]}')
        print('  Top LIME features:')
        for feat, weight in lime_exp.as_list()[:5]:
            print(f'    {feat:45s}: {weight:+.4f}')

except ImportError:
    print('\nLIME: install with pip install lime')
    print('LIME explains predictions by fitting a local linear model in the neighbourhood')
    print('of the instance being explained, perturbing input features.')

# ── SHAP vs LIME comparison ────────────────────────────────────────
print('\n=== SHAP vs LIME: Key Differences ===')
comparison = {
    'Mathematical Foundation': ['Game theory (Shapley values)', 'Local linear approximation'],
    'Consistency':             ['Guaranteed: always same answer', 'Stochastic: varies each run'],
    'Speed':                   ['Fast for trees (TreeExplainer)', 'Slow: requires many samples'],
    'Global Explanations':     ['Yes - aggregate SHAP values', 'No - local only'],
    'Faithfulness':            ['Exact for tree models', 'Approximate always'],
    'Best For':                ['Production; regulatory use', 'Quick local debugging'],
}
for aspect, (shap_val, lime_val) in comparison.items():
    print(f'  {aspect}:')
    print(f'    SHAP: {shap_val}')
    print(f'    LIME: {lime_val}')